### Phase 3: Gold Layer – Fraud Analytics

The Gold layer generates business-ready datasets used for fraud monitoring and analytics.

Key outputs:

-  Fraud alert detection
-  Account risk scoring
-  Simulated AI Fraus Detection

These datasets are optimized for fast analytical queries.


#### Load Silver

In [0]:
transfers = spark.table("silver_transfers_enriched")
balances = spark.table("silver_account_balance_cdc")

#### Fraud Rules

Flag suspicious transactions:

- amount > 10000
- txn_last_hour > 5
- account not verified

In [0]:
from pyspark.sql.functions import when, col

alerts = transfers.withColumn(
"suspicious",
when((col("amount")>10000) | (col("txn_last_hour")>5),1).otherwise(0)
)

alerts.filter("suspicious=1") \
.write.format("delta") \
.mode("overwrite") \
.saveAsTable("fact_fraud_alerts")

#### ->Account Risk Score

In [0]:
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import col, sum

risk = alerts.groupBy("account_id").agg(
sum("suspicious").alias("alert_count")
)

risk = risk.withColumn(
"risk_score",
col("alert_count")*10
)

risk.write.format("delta") \
.mode("overwrite") \
.saveAsTable("dim_account_risk_score")

#### ->Simulated AI Fraud Detection

In [0]:
anomaly = transfers.withColumn(
"anomaly_flag",
when(col("amount")>15000,1).otherwise(0)
)

anomaly.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_anomaly_transactions")